# project_10_ndm1_binder — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [1]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

Python : 3.11.15
Platform: Linux-6.18.5-x86_64-with-glibc2.39
GPU    : NONE FOUND
Structure prediction on CPU is impractically slow.


## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [2]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

Note: you may need to restart the kernel to use updated packages.
Core install done.


In [3]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

# Environment stamp 2026-06-24T03:15:38 UTC
Bio            1.84


py3Dmol        2.4.0


numpy          2.4.6


pandas         3.0.3
matplotlib     3.11.0


seaborn        0.13.2
tqdm           4.68.3
requests       2.33.1


## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [4]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

Helpers ready: install_colabfold(), install_esmfold().


## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [5]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

seeds set to 0
logged: Ran 00_setup; environment stamped.


## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [6]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

Uncomment to mount Drive and set your working directory.


---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — di-zinc target prep, active-site-rim hotspots, binder + occlusion metrics

**Standard slot:** *define & explore.* **For Project 10 this means:** clean **NDM-1** while
**preserving both catalytic Zn²⁺ ions**, select the **hotspot residues on the active-site rim** (the
walls of the substrate-access channel — the occluding epitope), write down the binder + occlusion
metrics + cutoffs, and run a deterministic **mock** mini-run as your "hello-world" (D0).

> **Defensive anti-AMR framing.** The goal is to **inhibit** NDM-1 so a last-resort antibiotic works
> again — *not* to enhance resistance or pathogen fitness. The binder occludes the substrate channel;
> it never stabilizes or protects the enzyme. See the Responsible Research sections of `README.md` /
> `MANUAL.md`.

Run `00_setup.ipynb` first in this session. A real binder campaign wants an **A100** (see
`MANUAL.md §2`); everything here runs on a no-GPU **mock** backend so you can build the plumbing
anywhere, then switch to the real backend on Colab Pro / A100.

## The binder + occlusion metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the binder | thermostability / ΔG |
| **pae_interaction** | Å | AF2-Multimer error across the **binder–target interface** (the key binder metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| rosetta_dG | REU | interface energy (more negative = stronger) | a guarantee it binds |
| shape complementarity | 0–1 | interface packing quality | epitope correctness |
| **occlusion** | 0–1 | fraction of the substrate-access channel blocked (mechanism) | **inhibition** (needs the kinetics assay) |
| **specificity** | 0–1 | NDM-1-selectivity vs human metalloenzymes (higher = safer) | proof of no off-target effect |
| TM-score | 0–1 | similarity to nearest known fold (<0.5 ≈ novel) | a pass/fail of correctness |

The shared `"binder"` cutoffs: **scRMSD ≤ 2.5, pLDDT ≥ 80, pae_interaction ≤ 10, rosetta_dG ≤ −30,
sc ≥ 0.6.** Project additions (notebook 04): **occlusion ≥ 0.5, specificity ≥ 0.5.** `pae_interaction`
is the single most important binder metric — but a low value is *confidence*, **not** affinity. And
**binding ≠ inhibition**: a passing, occluding design is a **hypothesis** until the nitrocefin /
carbapenem IC50 assay (notebook 05).

## Setup paths

In [7]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_10_ndm1_binder/notebooks


## 1 · Di-zinc target prep + active-site-rim hotspots

The design target is **NDM-1 with its di-zinc active site preserved**, and the hotspots are the
**active-site-rim** residues that wall the **substrate (carbapenem) access channel** — steering the
binder there is what makes it an *occluding inhibitor*, not just a sticker. Fetch the candidate
structures with `data/download_data.py` (3SPU / 4EYL — **verify on RCSB**), isolate the NDM-1 chain,
remove waters/buffer and any hydrolyzed-substrate ligand, **keep both Zn²⁺ ions as heteroatoms**, and
read the rim residues off the channel.

> **Do NOT** strip the Zn²⁺ ions or design over the Zn-coordinating His/Cys/Asp residues — the binder
> sits on the rim and occludes substrate access; it does not replace the metal ligands.

Below we just *declare* an EXAMPLE rim-hotspot set so the notebook runs end-to-end; **replace it with
the residues you derive from the actual di-zinc structure** (numbering depends on the PDB you verify).

In [8]:
import binder_tools as bt

TARGET = "NDM1"                      # cleaned NDM-1 with BOTH Zn2+ preserved (you produce this from 3SPU/4EYL)
# EXAMPLE active-site-rim hotspots — VERIFY/REPLACE from the cleaned di-zinc structure (data/README.md).
# These are placeholders so the plumbing runs; real numbering depends on the PDB chain you clean.
# They name the RIM that walls the substrate-access channel — NOT the Zn-coordinating ligand residues.
HOTSPOTS = bt.parse_hotspots("A120,A220,A228")   # EXAMPLE_DATA placeholder rim residues
print("target  :", TARGET, "(di-zinc active site — both Zn2+ preserved)")
print("hotspots:", HOTSPOTS, " (EXAMPLE active-site-rim residues — replace with your verified rim)")

target  : NDM1 (di-zinc active site — both Zn2+ preserved)
hotspots: ('A120', 'A220', 'A228')  (EXAMPLE active-site-rim residues — replace with your verified rim)


## 2 · Mock hello-world: a tiny two-paradigm mini-run

`scripts/binder_tools.py` exposes both paradigms behind one API:
`generate_binders_bindcraft(...)` and `generate_binders_rfdiffusion(...)` (the latter → **LigandMPNN**,
Zn-aware, on the real backend), plus `af2_multimer(...)` (the scorer). The **mock** backend is
deterministic and GPU-free so you can develop the plumbing. **Never report mock numbers as real** —
they are `SYNTHETIC` by construction, and there are **no fabricated IC50s** anywhere.

In [9]:
# A few designs from each paradigm, scored by mock AF2-Multimer. All numbers are SYNTHETIC.
bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=3, tool="mock")
rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=3, tool="mock")
bt.score_designs(bc, tool="mock")
bt.score_designs(rf, tool="mock")

d = bc[0]
print("example BindCraft design:")
print("  id   :", d.design_id)
print("  len  :", d.length, "aa")
print("  seq  :", d.sequence)
print("  pae_interaction =", d.pae_interaction, " scrmsd =", d.scrmsd,
      " sc =", d.shape_complementarity, " (SYNTHETIC)")
print("  synthetic flag  :", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.")

example BindCraft design:
  id   : EXAMPLE_DATA_bindcraft_0000
  len  : 51 aa
  seq  : VRYPVMIAGDIAQHNTGRNAGHETQWNKGRSFCMSTCDSFLWYALHEKQDN
  pae_interaction = 17.0  scrmsd = 1.49  sc = 0.89  (SYNTHETIC)
  synthetic flag  : True -> SYNTHETIC — mock backend, not a real design/prediction

Reminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.


## 3 · Occlusion proxy (does it block the substrate channel?) + rim coverage

A binder only *inhibits* if it **occludes** the substrate-access channel over the di-zinc site.
`occlusion_score()` combines rim coverage (`hotspot_overlap`, the fraction of rim hotspots contacted)
with a pocket-fit term — a teaching stand-in for the pocket-volume / docking analysis in notebook 04
and the inhibition assay in notebook 05. Higher ⇒ more likely to block (**not** a guarantee:
occlusion ≠ inhibition).

In [10]:
for b in bc[:3]:
    ov = bt.hotspot_overlap(b.contact_residues, HOTSPOTS)
    occ = bt.occlusion_score(b, HOTSPOTS, tool="mock")
    print(f"{b.design_id}: contacts {b.contact_residues} -> rim coverage = {ov}, "
          f"occlusion = {occ['occlusion']} (occludes? {occ['occludes']}) (SYNTHETIC)")
print("\nNOTE:", occ["note"])

EXAMPLE_DATA_bindcraft_0000: contacts ('A120',) -> rim coverage = 0.333, occlusion = 0.58 (occludes? True) (SYNTHETIC)
EXAMPLE_DATA_bindcraft_0001: contacts ('A120', 'A220', 'A228') -> rim coverage = 1.0, occlusion = 0.808 (occludes? True) (SYNTHETIC)
EXAMPLE_DATA_bindcraft_0002: contacts ('A120', 'A220', 'A228') -> rim coverage = 1.0, occlusion = 0.816 (occludes? True) (SYNTHETIC)

NOTE: SYNTHETIC occlusion proxy — occlusion is NOT inhibition; see notebook 05 assay


## 4 · Specificity proxy vs a human metalloenzyme (safety counter-test)

A di-zinc-site binder that also hits a **human** Zn/metalloenzyme (carbonic anhydrase, MMPs,
glyoxalase II) is a safety liability. `offtarget_specificity()` returns a 0–1 selectivity (higher =
more NDM-1-selective). This is the in-silico mirror of the off-target-metalloenzyme **control** in the
inhibition assay (notebook 05). SYNTHETIC on the mock backend.

In [11]:
spec = bt.offtarget_specificity(bc[0], "CA2", tool="mock")   # CA2 = human carbonic anhydrase II
print(f"{bc[0].design_id} vs human {spec['off_target']}: specificity = {spec['specificity']} "
      f"(selective? {spec['selective']}) (SYNTHETIC)")
print("NOTE:", spec["note"])

EXAMPLE_DATA_bindcraft_0000 vs human CA2: specificity = 0.824 (selective? True) (SYNTHETIC)
NOTE: SYNTHETIC specificity proxy — confirm with a real off-target panel on Colab


## Visualize a binder–target complex (py3Dmol)

Use this to eyeball a predicted binder–NDM-1 complex once you have a real PDB (from AF2-Multimer) —
and to confirm the binder sits **over the di-zinc substrate-access rim**, with both Zn²⁺ retained.

In [12]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.addStyle({"resn": "ZN"}, {"sphere": {"color": "grey", "radius": 0.6}})  # show the di-zinc site
    view.zoomTo()
    return view.show()

# Example (after a real AF2-Multimer prediction writes a complex PDB):
# show_complex("results/af2/top_complex.pdb")
print("show_complex(pdb_path) ready (renders the di-zinc ions as spheres).")

show_complex(pdb_path) ready (renders the di-zinc ions as spheres).


## D0 checklist
- [ ] NDM-1 accessions verified on RCSB (3SPU/4EYL are candidates); chain identified; **both Zn²⁺ present**.
- [ ] Cleaned di-zinc target (Zn preserved) + **active-site-rim hotspot list** (the substrate-channel walls, not invented, not the Zn ligands).
- [ ] One-paragraph definition of each binder/occlusion metric **with** its "does not mean" note (esp. binding ≠ inhibition).
- [ ] Reproduced mock mini-run (both paradigms) with metrics + occlusion + specificity printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria (incl. an occlusion threshold) + controls; `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the two-paradigm binder campaign at the active-site rim.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — two-paradigm binder design vs the NDM-1 active-site rim

**Standard slot:** *design campaign.* **For Project 10 this is the core:** run **both** paradigms
against the **active-site-rim** hotspots (di-zinc preserved) and assemble their pools (D2):
- **BindCraft** (one-shot hallucination, AF2-Multimer in the loop) — **50–200** designs.
- **RFdiffusion binder mode → LigandMPNN** (Zn-aware near the metal) — **500–1000** backbones → sequences.

Then score every design with **AF2-Multimer** (`pae_interaction` is the key binder metric).

> **Compute honesty:** a real campaign at this scale wants an **A100** (Colab Pro+ or a cluster).
> Free **T4** = a *small fallback* (FreeBindCraft, small `num_designs`, a small RFdiffusion batch +
> ESMFold triage). The cells below run on the deterministic **mock** backend so the plumbing executes
> anywhere; the real calls + A100 notes are shown alongside. Run `00_setup.ipynb` first.

> **Defensive-anti-AMR reminder:** keep the di-zinc site intact and the binder framed as an
> **inhibitor / β-lactam adjuvant** — never a tool to enhance resistance or pathogen fitness.

## Setup paths

In [13]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_10_ndm1_binder/notebooks


## Version-verify the pinned upstreams (tools change!)

The binder tools live in fast-moving upstream repos. **Pin commits** and **verify the URLs still
exist** before relying on them (`requests.head`; a non-200 means it moved — update the pin and log
it). Note **LigandMPNN** here (the Zn-aware sequence designer — the key swap vs the PD-L1 template).
The generation itself needs an A100; this check needs nothing.

In [14]:
import requests

# Pinned upstreams (pin a COMMIT/tag in your repo — these change; commit hashes go in comments):
#   BindCraft      https://github.com/martinpacesa/BindCraft        # e.g. pin <commit>
#   FreeBindCraft  https://github.com/cytokineking/FreeBindCraft     # free-tier fallback — VERIFY it exists; pin <commit>
#   RFdiffusion    https://github.com/RosettaCommons/RFdiffusion     # pin <commit>
#   LigandMPNN     https://github.com/dauparas/LigandMPNN            # Zn-AWARE sequence design near the di-zinc site; pin <commit>
#   ColabFold      https://github.com/sokrypton/ColabFold            # AF2-Multimer; pin <commit>
PINNED = {
    "BindCraft":     "https://github.com/martinpacesa/BindCraft",
    "FreeBindCraft": "https://github.com/cytokineking/FreeBindCraft",
    "RFdiffusion":   "https://github.com/RosettaCommons/RFdiffusion",
    "LigandMPNN":    "https://github.com/dauparas/LigandMPNN",
    "ColabFold":     "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:14s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:14s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")
print("FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.")

  [200] BindCraft      https://github.com/martinpacesa/BindCraft


  [200] FreeBindCraft  https://github.com/cytokineking/FreeBindCraft


  [200] RFdiffusion    https://github.com/RosettaCommons/RFdiffusion


  [200] LigandMPNN     https://github.com/dauparas/LigandMPNN


  [200] ColabFold      https://github.com/sokrypton/ColabFold

Non-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.
FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.


## 1 · Define the campaign

Same target + active-site-rim hotspots as notebook 01. Set honest campaign sizes; the cells run on
`mock` so they execute anywhere. On Colab (A100) switch `TOOL_*` to the real backends — and **shrink
the numbers on a T4** (FreeBindCraft, a small RFdiffusion batch).

In [15]:
import binder_tools as bt
import pandas as pd

TARGET = "NDM1"
HOTSPOTS = bt.parse_hotspots("A120,A220,A228")   # EXAMPLE active-site-rim residues — replace with your verified rim

# Honest campaign sizes (catalog): BindCraft 50-200, RFdiffusion 500-1000 backbones.
# We use small mock counts here so the dry run is fast; scale up with the real backend on A100.
N_BINDCRAFT   = 60      # -> 50-200 on A100; fewer (FreeBindCraft) on T4
N_RFDIFFUSION = 200     # -> 500-1000 backbones on A100; small batch on T4

TOOL_BINDCRAFT   = "mock"   # -> "bindcraft" / "freebindcraft" on Colab
TOOL_RFDIFFUSION = "mock"   # -> "rfdiffusion" on Colab (RFdiffusion binder mode -> LigandMPNN, Zn-aware)
TOOL_AF2         = "mock"   # -> "af2" (ColabFold AF2-Multimer) on Colab

print(f"BindCraft   : n={N_BINDCRAFT}  tool={TOOL_BINDCRAFT}")
print(f"RFdiffusion : n={N_RFDIFFUSION} tool={TOOL_RFDIFFUSION} (-> LigandMPNN, Zn-aware on Colab)")
print(f"AF2-Multimer: tool={TOOL_AF2}")
print("hotspots    :", HOTSPOTS, "(active-site rim; di-zinc preserved)")

BindCraft   : n=60  tool=mock
RFdiffusion : n=200 tool=mock (-> LigandMPNN, Zn-aware on Colab)
AF2-Multimer: tool=mock
hotspots    : ('A120', 'A220', 'A228') (active-site rim; di-zinc preserved)


## 2 · Paradigm #1 — BindCraft campaign

One-shot hallucination with AF2-Multimer in the loop. On A100 this produces 50–200 binders
pre-filtered on interface confidence; we still re-score with AF2-Multimer so the head-to-head with
RFdiffusion is apples-to-apples. Keep the di-zinc target intact (AF2/BindCraft don't place Zn²⁺ — keep
it as a heteroatom). The `mock` backend returns deterministic `SYNTHETIC` designs.

In [16]:
# Real call (Colab, A100): bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT, tool="bindcraft")
#   free-tier fallback: tool="freebindcraft", smaller N. See MANUAL.md §2 / scripts/binder_tools.py TODOs.
bindcraft = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT, tool=TOOL_BINDCRAFT)
bt.score_designs(bindcraft, tool=TOOL_AF2)     # AF2-Multimer -> pae_interaction, plddt, scrmsd, sc
print(f"BindCraft pool: {len(bindcraft)} designs (tool={TOOL_BINDCRAFT}; SYNTHETIC if mock)")
print("example:", bindcraft[0].design_id, "pae_interaction=", bindcraft[0].pae_interaction)

BindCraft pool: 60 designs (tool=mock; SYNTHETIC if mock)
example: EXAMPLE_DATA_bindcraft_0000 pae_interaction= 17.0


## 3 · Paradigm #2 — RFdiffusion binder campaign → LigandMPNN (Zn-aware)

Diffuse binder backbones docked at the active-site-rim hotspots, then **LigandMPNN** designs sequences
**aware of the di-zinc ligand context** (Dauparas 2024 — *not* vanilla ProteinMPNN), then AF2-Multimer
re-predicts each complex. On A100 this is 500–1000 backbones (the per-backbone hit rate is low — that
is normal). The `mock` backend stands in for the whole chain.

In [17]:
# Real call (Colab, A100): bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION,
#   tool="rfdiffusion", mpnn_temperature=0.1, num_seq_per_backbone=8). LigandMPNN passes the Zn ions;
#   AF2-Multimer is the slow step.
rfdiff = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION, tool=TOOL_RFDIFFUSION)
bt.score_designs(rfdiff, tool=TOOL_AF2)
print(f"RFdiffusion pool: {len(rfdiff)} designs (tool={TOOL_RFDIFFUSION}; SYNTHETIC if mock)")
print("example:", rfdiff[0].design_id, "pae_interaction=", rfdiff[0].pae_interaction)

RFdiffusion pool: 200 designs (tool=mock; SYNTHETIC if mock)
example: EXAMPLE_DATA_rfdiffusion_0000 pae_interaction= 9.0


## 4 · Assemble + persist both pools

Write one tidy CSV per paradigm (plus a combined one). These feed notebook 03 (the shared filter) and
notebook 04 (occlusion + specificity). We add an EXAMPLE physics column (`rosetta_dG`) and a rim-
coverage column here so the binder physics layer and the occlusion analysis have something to act on
in the dry run — on Colab these come from FreeBindCraft/PyRosetta and the pocket analysis; for `mock`
they are SYNTHETIC.

In [18]:
import pandas as pd

def pool_to_df(designs):
    rows = []
    for d in designs:
        # In the mock dry run we attach an EXAMPLE_DATA interface energy so Layer 3 (physics) is
        # exercised. On Colab, replace with the real FreeBindCraft/PyRosetta rosetta_dG + solubility.
        rdg = -45.0 + (bt._hashints("dG", d.design_id) % 40)   # SYNTHETIC, range ~ -45..-6 REU
        occ = bt.occlusion_score(d, d.hotspots, tool="mock")    # SYNTHETIC occlusion proxy (mechanism)
        rows.append(dict(
            design_id=d.design_id, paradigm=d.paradigm, target=d.target,
            length=d.length, sequence=d.sequence,
            plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
            shape_complementarity=d.shape_complementarity,
            rosetta_dG=round(float(rdg), 2), solubility=0.3,
            contact_residues=",".join(d.contact_residues),
            hotspot_overlap=bt.hotspot_overlap(d.contact_residues, d.hotspots),
            occlusion=occ["occlusion"],          # SYNTHETIC — occlusion is NOT inhibition (assay tests that)
            synthetic=d.synthetic,
        ))
    return pd.DataFrame(rows)

df_bc = pool_to_df(bindcraft); df_bc.to_csv("results/bindcraft_designs.csv", index=False)
df_rf = pool_to_df(rfdiff);    df_rf.to_csv("results/rfdiffusion_designs.csv", index=False)
combined = pd.concat([df_bc, df_rf], ignore_index=True)
combined.to_csv("results/all_designs.csv", index=False)

print("wrote results/bindcraft_designs.csv   ", df_bc.shape)
print("wrote results/rfdiffusion_designs.csv ", df_rf.shape)
print("wrote results/all_designs.csv         ", combined.shape)
print("\nALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.")
print("Occlusion is a STRUCTURAL proxy; binding/occlusion != inhibition (the IC50 comes only from nb 05).")
combined.head(4)

wrote results/bindcraft_designs.csv    (60, 15)
wrote results/rfdiffusion_designs.csv  (200, 15)
wrote results/all_designs.csv          (260, 15)

ALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.
Occlusion is a STRUCTURAL proxy; binding/occlusion != inhibition (the IC50 comes only from nb 05).


,design_id,paradigm,target,length,sequence,plddt,pae_interaction,scrmsd,shape_complementarity,rosetta_dG,solubility,contact_residues,hotspot_overlap,occlusion,synthetic
0,EXAMPLE_DATA_bindcraft_0000,bindcraft,NDM1,51,VRYPVMIAGDIAQHNTGRNAGHETQWNKGRSFCMSTCDSFLWYALH...,99.0,17.0,1.49,0.89,-38.0,0.3,A120,0.333,0.580,True
1,EXAMPLE_DATA_bindcraft_0001,bindcraft,NDM1,49,DNKLWITVHIAGMSFQHNKCMNKGDNACMETQMSKLDETVMYPCWYTQW,70.0,4.0,2.70,0.60,-19.0,0.3,"A120,A220,A228",1.000,0.808,True
2,EXAMPLE_DATA_bindcraft_0002,bindcraft,NDM1,47,PCWEAQMNFCWIFQWEFCDNTQRIALWNTCDYACRIKQRIPLWEPCH,98.0,4.0,1.08,0.58,-35.0,0.3,"A120,A220,A228",1.000,0.816,True
3,EXAMPLE_DATA_bindcraft_0003,bindcraft,NDM1,61,EPLWNFCHSPQHYPCWYTQHNTVRSPVWEPQDIPQDYFGHIALMET...,90.0,16.0,2.30,0.50,-38.0,0.3,A120,0.333,0.584,True


## D2 checklist
- [ ] BindCraft pool generated at honest scale (50–200 on A100; FreeBindCraft/small on T4); di-zinc kept.
- [ ] RFdiffusion-binder pool generated (500–1000 backbones → **LigandMPNN**, Zn-aware, on A100).
- [ ] Every design scored by AF2-Multimer (`pae_interaction` parsed); both pools written to `results/`.
- [ ] Design log: every config + seed + tool **commit** + output path, in `LOG.md`.
- [ ] Version-verify output captured; 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** filter on both pools.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer filter (binder cutoffs)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 10** you build `fp.Design` **binder** objects from both pools, call
`fp.run_pipeline(..., design_type="binder")`, and `fp.report(...)` the survival funnel + ranked CSV,
**per paradigm** so the head-to-head is fair (D3 part 1). The project-specific **occlusion** and
**specificity** thresholds are applied in notebook 04 (they are not part of the shared module).

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back. This notebook *imports* it.

Run `00`–`02` first so `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` exist.

## Setup paths

In [19]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_10_ndm1_binder/notebooks


## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"binder"` cutoffs: scRMSD ≤ 2.5, pLDDT ≥ 80, pae ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6. (Occlusion +
specificity are project-specific and applied in notebook 04.)

In [20]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])

Loaded shared filtering_pipeline from: /home/user/biofx_python/denovo_protein_design_course/shared/filtering_pipeline.py
binder cutoffs: {'scrmsd': 2.5, 'plddt': 80, 'pae': 10, 'rosetta_dG': -30, 'sc': 0.6}


## Build `Design` (binder) objects from the pools

Map each pool row onto `fp.Design` with `design_type="binder"`. The binder metrics drive the layers:
`scrmsd`/`plddt`/`pae_interaction` (Layer 1 self-consistency), and
`rosetta_dG`/`shape_complementarity`/`solubility` (Layer 3 physics). We keep `paradigm`,
`hotspot_overlap`, and `occlusion` in `extra` for the occlusion + specificity + head-to-head analysis
in notebook 04. (Mock has no independent orthogonal predictor, so we run Layers 1+3 here; on Colab add
a second predictor for Layer 2.)

In [21]:
import os
import pandas as pd

# Regenerate the pools if a fresh session lost them (deterministic mock).
if not (os.path.exists("results/bindcraft_designs.csv") and os.path.exists("results/rfdiffusion_designs.csv")):
    import binder_tools as bt
    TARGET, HOTSPOTS = "NDM1", bt.parse_hotspots("A120,A220,A228")
    bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=60, tool="mock");  bt.score_designs(bc, tool="mock")
    rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=200, tool="mock"); bt.score_designs(rf, tool="mock")
    def _q(designs, p):
        rows=[dict(design_id=d.design_id, paradigm=d.paradigm, length=d.length, sequence=d.sequence,
                   plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
                   shape_complementarity=d.shape_complementarity,
                   rosetta_dG=round(-45.0+(bt._hashints("dG",d.design_id)%40),2), solubility=0.3,
                   hotspot_overlap=bt.hotspot_overlap(d.contact_residues,d.hotspots),
                   occlusion=bt.occlusion_score(d, d.hotspots, tool="mock")["occlusion"],
                   synthetic=d.synthetic)
              for d in designs]
        pd.DataFrame(rows).to_csv(p, index=False)
    _q(bc, "results/bindcraft_designs.csv"); _q(rf, "results/rfdiffusion_designs.csv")

df_bc = pd.read_csv("results/bindcraft_designs.csv")
df_rf = pd.read_csv("results/rfdiffusion_designs.csv")

def row_to_binder(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type="binder",
        plddt=r.get("plddt"), pae_interaction=r.get("pae_interaction"), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd predictor on Colab
        rosetta_dG=r.get("rosetta_dG"), shape_complementarity=r.get("shape_complementarity"),
        solubility=r.get("solubility", 0.3),
        extra={"paradigm": r.get("paradigm"), "hotspot_overlap": r.get("hotspot_overlap"),
               "occlusion": r.get("occlusion")},
    )

binders_bc = [row_to_binder(r) for _, r in df_bc.iterrows()]
binders_rf = [row_to_binder(r) for _, r in df_rf.iterrows()]
print(f"built {len(binders_bc)} BindCraft + {len(binders_rf)} RFdiffusion binder Designs")

built 60 BindCraft + 200 RFdiffusion binder Designs


## Run the pipeline — per paradigm (fair head-to-head)

`run_pipeline(design_type="binder")` applies the binder cutoffs in order and returns a ranked
DataFrame with survival counts in `df.attrs`. We run **each paradigm separately** so the
survival-at-each-layer funnels are comparable. We use Layers 1+3 here (mock has no independent
orthogonal source; add Layer 2 on Colab with a second predictor).

In [22]:
def run_one(designs, label):
    df = fp.run_pipeline(designs, design_type="binder", use_layers=(1, 3))
    df["paradigm"] = label
    surv = df.attrs["survival"]; n = df.attrs["n_total"]
    passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: {n} designs, survival {surv}, all-layers hit rate = {passed}/{n} ({100*passed/max(n,1):.1f}%)")
    return df

ranked_bc = run_one(binders_bc, "bindcraft")
ranked_rf = run_one(binders_rf, "rfdiffusion")

ranked = pd.concat([ranked_bc, ranked_rf], ignore_index=True).sort_values(
    ["layers_passed", "score"], ascending=False).reset_index(drop=True)
ranked.to_csv("results/all_ranked.csv", index=False)
print("\nwrote results/all_ranked.csv", ranked.shape)
ranked.head(10)[["design_id", "paradigm", "layers_passed", "score",
                 "scrmsd", "plddt", "pae_interaction", "rosetta_dG"]]

bindcraft   : 60 designs, survival {'L1': 9, 'L3': 0}, all-layers hit rate = 0/60 (0.0%)
rfdiffusion : 200 designs, survival {'L1': 33, 'L3': 9}, all-layers hit rate = 9/200 (4.5%)

wrote results/all_ranked.csv (260, 21)


,design_id,paradigm,layers_passed,score,scrmsd,plddt,pae_interaction,rosetta_dG
0,EXAMPLE_DATA_rfdiffusion_0184,rfdiffusion,3,4.5633,0.88,98.0,8.0,-44.0
1,EXAMPLE_DATA_rfdiffusion_0099,rfdiffusion,3,4.2767,1.24,84.0,4.0,-38.0
2,EXAMPLE_DATA_rfdiffusion_0171,rfdiffusion,3,3.9400,1.07,87.0,9.0,-32.0
3,EXAMPLE_DATA_rfdiffusion_0033,rfdiffusion,3,3.8367,1.24,94.0,10.0,-36.0
4,EXAMPLE_DATA_rfdiffusion_0075,rfdiffusion,3,3.4900,1.98,88.0,6.0,-42.0
5,EXAMPLE_DATA_rfdiffusion_0198,rfdiffusion,3,3.2767,1.78,98.0,10.0,-33.0
6,EXAMPLE_DATA_rfdiffusion_0034,rfdiffusion,3,3.1033,2.06,96.0,8.0,-31.0
7,EXAMPLE_DATA_rfdiffusion_0173,rfdiffusion,3,2.9300,2.40,90.0,6.0,-34.0
8,EXAMPLE_DATA_rfdiffusion_0148,rfdiffusion,3,2.7467,2.41,81.0,7.0,-34.0
9,EXAMPLE_DATA_rfdiffusion_0138,rfdiffusion,1,4.0700,0.92,92.0,6.0,-41.0


## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Here we report the
**combined** pool for one comparable figure; the per-paradigm runs above are the rigorous version.
Read the bars as a funnel: steep drops show which layer discriminates.

In [23]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

all_binders = binders_bc + binders_rf
df_all = fp.run_pipeline(all_binders, design_type="binder", use_layers=(1, 3))
top = fp.report(df_all, top_n=15, save_prefix="results/p10")
print("\nsaved results/p10_survival.png + results/p10_ranked.csv")
top

Total designs: 260
  L1 survivors: 42  (16.2%)
  L3 survivors: 9  (3.5%)



saved results/p10_survival.png + results/p10_ranked.csv


,design_id,design_type,layers_passed,score,scrmsd,plddt,pae_interaction,rosetta_dG,tm_to_pdb
0,EXAMPLE_DATA_rfdiffusion_0184,binder,3,4.5633,0.88,98.0,8.0,-44.0,None
1,EXAMPLE_DATA_rfdiffusion_0099,binder,3,4.2767,1.24,84.0,4.0,-38.0,None
2,EXAMPLE_DATA_rfdiffusion_0171,binder,3,3.9400,1.07,87.0,9.0,-32.0,None
3,EXAMPLE_DATA_rfdiffusion_0033,binder,3,3.8367,1.24,94.0,10.0,-36.0,None
4,EXAMPLE_DATA_rfdiffusion_0075,binder,3,3.4900,1.98,88.0,6.0,-42.0,None
5,EXAMPLE_DATA_rfdiffusion_0198,binder,3,3.2767,1.78,98.0,10.0,-33.0,None
6,EXAMPLE_DATA_rfdiffusion_0034,binder,3,3.1033,2.06,96.0,8.0,-31.0,None
7,EXAMPLE_DATA_rfdiffusion_0173,binder,3,2.9300,2.40,90.0,6.0,-34.0,None
8,EXAMPLE_DATA_rfdiffusion_0148,binder,3,2.7467,2.41,81.0,7.0,-34.0,None
9,EXAMPLE_DATA_rfdiffusion_0138,binder,1,4.0700,0.92,92.0,6.0,-41.0,None


## Honest hit-rate accounting (per paradigm)

Report `N passing all layers / N generated` for **each** paradigm — this is the number the occlusion +
specificity + head-to-head analysis in notebook 04 builds on. Remember: survival is *enrichment*, not
*correctness*, and a passing binder is **not** a measured inhibitor. Mock numbers are SYNTHETIC.

In [24]:
for label, df in [("bindcraft", ranked_bc), ("rfdiffusion", ranked_rf)]:
    n = len(df); passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: layers_passed distribution {df['layers_passed'].value_counts().sort_index().to_dict()}")
    print(f"{'':12s}  all-layers survivors = {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")

bindcraft   : layers_passed distribution {0: 51, 1: 9}
              all-layers survivors = 0/60 (0.0%)  [SYNTHETIC if mock]
rfdiffusion : layers_passed distribution {0: 167, 1: 24, 3: 9}
              all-layers survivors = 9/200 (4.5%)  [SYNTHETIC if mock]


## D3 (part 1) checklist
- [ ] `results/all_ranked.csv` produced by the **shared** module (`design_type="binder"`), not a one-off script.
- [ ] Survival-at-each-layer reported **per paradigm** (funnel figure `results/p10_survival.png`).
- [ ] Honest hit-rate accounting (N pass / N generated) for BindCraft and RFdiffusion.
- [ ] Mapping assumptions (which fields → which `Design` attributes; occlusion kept in `extra`) written down.

**Next:** `04_validate.ipynb` — occlusion modeling + specificity vs human metalloenzymes + the head-to-head.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — occlusion + specificity vs human metalloenzymes + BindCraft-vs-RFdiffusion

**Standard slot:** *validate (in silico).* **For Project 10 this is the mechanism core:** model
**substrate occlusion** (does the binder block the carbapenem-access channel?), counter-test
**specificity vs human metalloenzymes** (safety), run the **head-to-head** between the two paradigms,
and the **rim-vs-distal epitope ablation** (a distal patch should bind but **not** occlude), with
publication-style figures (D3 part 2).

> **Binding ≠ inhibition.** Occlusion and specificity are in-silico *enrichment*. Only the
> nitrocefin/carbapenem kinetics assay (notebook 05) measures inhibition (IC50). No IC50 is produced
> here — that would be fabricated.

Needs `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` + `results/all_ranked.csv`
(from notebooks 02–03).

## Setup paths

In [25]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_10_ndm1_binder/notebooks


## 1 · Head-to-head hit rate + interface energy

Compare the two paradigms on (a) all-layers **hit rate** and (b) the **interface-energy** (`rosetta_dG`)
distribution of survivors. A fair comparison filters both identically (notebook 03) and reports the
*distribution*, not the single best. Mock numbers are SYNTHETIC.

In [26]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
print("paradigms:", ranked["paradigm"].value_counts().to_dict())

summary = []
for p, g in ranked.groupby("paradigm"):
    n = len(g); passed = int((g["layers_passed"] >= 3).sum())
    summary.append(dict(paradigm=p, n=n, all_layers_survivors=passed,
                        hit_rate_pct=round(100*passed/max(n,1), 1),
                        median_pae=round(float(g["pae_interaction"].median()), 2),
                        median_dG=round(float(g["rosetta_dG"].median()), 2)))
summary = pd.DataFrame(summary)
print("\nhead-to-head summary (SYNTHETIC if mock):")
print(summary.to_string(index=False))

paradigms: {'rfdiffusion': 200, 'bindcraft': 60}

head-to-head summary (SYNTHETIC if mock):
   paradigm   n  all_layers_survivors  hit_rate_pct  median_pae  median_dG
  bindcraft  60                     0           0.0        11.5      -31.0
rfdiffusion 200                     9           4.5        11.0      -26.5


In [27]:
# Interface-energy distribution per paradigm (survivors).
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for p, g in ranked.groupby("paradigm"):
    surv = g[g["layers_passed"] >= 3]
    ax[0].hist(g["pae_interaction"].dropna(), bins=15, alpha=0.5, label=p)
    ax[1].hist(surv["rosetta_dG"].dropna(), bins=15, alpha=0.5, label=p)
ax[0].set_xlabel("pae_interaction (Å, lower better)"); ax[0].set_ylabel("designs"); ax[0].set_title("AF2-Multimer pae_interaction"); ax[0].legend()
ax[1].set_xlabel("rosetta_dG (REU, more negative better)"); ax[1].set_title("Interface energy (survivors)"); ax[1].legend()
fig.suptitle("BindCraft vs RFdiffusion vs NDM-1 (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p10_headtohead.png", dpi=150); plt.show()
print("saved results/p10_headtohead.png")

saved results/p10_headtohead.png


## 2 · Substrate-occlusion modeling (mechanism #1) `[core]`

A binder only **inhibits** if it occludes the substrate-access channel over the di-zinc site. We
re-score every survivor with `occlusion_score()` (the mock proxy combines rim coverage with a
pocket-fit term; on Colab, replace with a pocket-volume / SASA with-vs-without-binder measurement and
optional carbapenem docking — see `binder_tools.occlusion_score(..., tool="pocket")`). Compare the
occlusion distributions across paradigms; an interface that does **not** occlude is a sticker, not an
inhibitor.

In [28]:
import binder_tools as bt

bc = pd.read_csv("results/bindcraft_designs.csv")
rf = pd.read_csv("results/rfdiffusion_designs.csv")
pools = pd.concat([bc, rf], ignore_index=True)

# Join precomputed occlusion (from nb 02) onto the ranked survivors; recompute if absent.
if "occlusion" in pools.columns:
    occ_map = pools.set_index("design_id")["occlusion"]
    ranked["occlusion"] = ranked["design_id"].map(occ_map)
else:
    HOTSPOTS = bt.parse_hotspots("A120,A220,A228")
    ranked["occlusion"] = ranked.apply(
        lambda r: bt.occlusion_score(r.get("contact_residues", ""), HOTSPOTS, tool="mock")["occlusion"], axis=1)

OCCLUSION_MIN = 0.5      # project threshold: occludes >= half the substrate-access channel
surv = ranked[ranked["layers_passed"] >= 3].copy()
print("occlusion of all-layers survivors (SYNTHETIC if mock):")
for p, g in surv.groupby("paradigm"):
    occluders = int((g["occlusion"] >= OCCLUSION_MIN).sum())
    print(f"  {p:12s}: median occlusion = {g['occlusion'].median():.2f}  | occluders(>= {OCCLUSION_MIN}) = {occluders}/{len(g)}")

fig, ax = plt.subplots(figsize=(5.2, 3.4))
for p, g in surv.groupby("paradigm"):
    ax.hist(g["occlusion"].dropna(), bins=12, alpha=0.5, label=p)
ax.axvline(OCCLUSION_MIN, color="k", ls="--", lw=1, label=f"threshold {OCCLUSION_MIN}")
ax.set_xlabel("occlusion (0–1, higher = more channel blocked)"); ax.set_ylabel("survivors")
ax.set_title("Substrate occlusion (EXAMPLE_DATA if mock)"); ax.legend()
plt.tight_layout(); plt.savefig("results/p10_occlusion.png", dpi=150); plt.show()
print("saved results/p10_occlusion.png  —  occlusion != inhibition (the IC50 assay tests that, nb 05)")

occlusion of all-layers survivors (SYNTHETIC if mock):
  rfdiffusion : median occlusion = 0.53  | occluders(>= 0.5) = 5/9


saved results/p10_occlusion.png  —  occlusion != inhibition (the IC50 assay tests that, nb 05)


## 3 · Specificity vs human metalloenzymes (mechanism #2) `[core]`

A di-zinc-site binder that also hits **human** Zn/metalloenzymes is a safety liability. Counter-test
each survivor against a small panel with `offtarget_specificity()` (on Colab: AF2-Multimer of the
binder vs each human enzyme; here a deterministic SYNTHETIC proxy). Higher specificity = more
NDM-1-selective. This is the in-silico mirror of the off-target-metalloenzyme **control** in the
assay (notebook 05).

In [29]:
HUMAN_PANEL = ["CA2", "MMP9", "GLO2"]   # carbonic anhydrase II, MMP-9, glyoxalase II (human metalloenzymes)
SPECIFICITY_MIN = 0.5

# Build a tiny BinderDesign-like shim from each survivor row so offtarget_specificity() can read .sequence.
spec_rows = []
for _, r in surv.iterrows():
    seq = str(r.get("sequence", "")) or "A"
    # Worst-case (minimum) specificity across the panel = most conservative safety read.
    svals = []
    for enz in HUMAN_PANEL:
        # pass a lightweight object exposing sequence + design_id via a namespace
        shim = type("D", (), {"sequence": seq, "design_id": str(r["design_id"]), "pae_interaction": r.get("pae_interaction")})()
        svals.append(bt.offtarget_specificity(shim, enz, tool="mock")["specificity"])
    spec_rows.append(dict(design_id=str(r["design_id"]), paradigm=r["paradigm"],
                          min_specificity=round(min(svals), 3)))
spec_df = pd.DataFrame(spec_rows)
surv = surv.merge(spec_df[["design_id", "min_specificity"]], on="design_id", how="left")

print(f"specificity (min across {HUMAN_PANEL}) of survivors (SYNTHETIC if mock):")
for p, g in surv.groupby("paradigm"):
    selective = int((g["min_specificity"] >= SPECIFICITY_MIN).sum())
    print(f"  {p:12s}: median min-specificity = {g['min_specificity'].median():.2f} | selective(>= {SPECIFICITY_MIN}) = {selective}/{len(g)}")
print("\nHigh specificity = LOW predicted off-target binding to human metalloenzymes (safer).")

specificity (min across ['CA2', 'MMP9', 'GLO2']) of survivors (SYNTHETIC if mock):
  rfdiffusion : median min-specificity = 0.18 | selective(>= 0.5) = 1/9

High specificity = LOW predicted off-target binding to human metalloenzymes (safer).


## 4 · Rim-vs-distal epitope ablation `[extension]`

The catalog ablation: **active-site rim vs a distal patch.** A binder steered to a **distal** surface
patch should still bind (decent `pae_interaction`) but **not** occlude the substrate channel (low
`occlusion`) — the cleanest demonstration that *epitope choice drives inhibition*. Here we scaffold it
by generating a distal-hotspot mock pool and comparing occlusion; on Colab, re-run the campaign with a
distal hotspot set and compare.

In [30]:
# Scaffold: a distal-patch pool (different, non-active-site hotspots) for the ablation.
RIM_HOTSPOTS    = bt.parse_hotspots("A120,A220,A228")    # active-site rim (the inhibitory epitope)
DISTAL_HOTSPOTS = bt.parse_hotspots("A40,A55,A70")        # EXAMPLE distal patch — replace with a real distal surface

rim_pool    = bt.generate_binders_bindcraft("NDM1", RIM_HOTSPOTS, n=40, tool="mock")
distal_pool = bt.generate_binders_bindcraft("NDM1", DISTAL_HOTSPOTS, n=40, tool="mock")
bt.score_designs(rim_pool, tool="mock"); bt.score_designs(distal_pool, tool="mock")

def median_occ(designs, active_site):
    vals = [bt.occlusion_score(d, active_site, tool="mock")["occlusion"] for d in designs]
    return float(np.median(vals))

# Both are scored for occlusion AGAINST THE ACTIVE-SITE RIM (the thing that must be blocked to inhibit).
print("rim-vs-distal ablation (occlusion scored vs the active-site rim; SYNTHETIC):")
print(f"  rim-targeted binders   : median occlusion = {median_occ(rim_pool, RIM_HOTSPOTS):.2f}  (expected HIGHER)")
print(f"  distal-targeted binders: median occlusion = {median_occ(distal_pool, RIM_HOTSPOTS):.2f}  (expected LOWER)")
print("\nInterpretation: distal binders may stick but should NOT occlude the substrate channel ->")
print("epitope choice (active-site rim) is what makes a binder an inhibitor candidate, not just a sticker.")

rim-vs-distal ablation (occlusion scored vs the active-site rim; SYNTHETIC):
  rim-targeted binders   : median occlusion = 0.66  (expected HIGHER)
  distal-targeted binders: median occlusion = 0.19  (expected LOWER)

Interpretation: distal binders may stick but should NOT occlude the substrate channel ->
epitope choice (active-site rim) is what makes a binder an inhibitor candidate, not just a sticker.


## 5 · Select the top 10–20 per paradigm (occluding + selective)

The D★ deliverable wants the **top 10–20 each**. Rank survivors by the composite score and, as
mechanism tie-breakers, prefer **higher occlusion** then **higher specificity** — an inhibitor
candidate must both block the channel and spare human metalloenzymes. Save the shortlist for the
inhibition-assay plan (notebook 05).

In [31]:
top_per = []
for p, g in surv.groupby("paradigm"):
    g2 = g.sort_values(["score", "occlusion", "min_specificity"], ascending=False).head(20)
    top_per.append(g2)
top = pd.concat(top_per, ignore_index=True)
top.to_csv("results/top_candidates.csv", index=False)
print("wrote results/top_candidates.csv:", top.shape, "(top<=20 per paradigm, occluding + selective)")
print(top.groupby("paradigm").size().to_dict())
cols = [c for c in ["design_id","paradigm","score","pae_interaction","rosetta_dG","occlusion","min_specificity"] if c in top.columns]
top.head(8)[cols]

wrote results/top_candidates.csv: (9, 23) (top<=20 per paradigm, occluding + selective)
{'rfdiffusion': 9}


,design_id,paradigm,score,pae_interaction,rosetta_dG,occlusion,min_specificity
0,EXAMPLE_DATA_rfdiffusion_0184,rfdiffusion,4.5633,8.0,-44.0,0.548,0.176
1,EXAMPLE_DATA_rfdiffusion_0099,rfdiffusion,4.2767,4.0,-38.0,0.208,0.000
2,EXAMPLE_DATA_rfdiffusion_0171,rfdiffusion,3.9400,9.0,-32.0,0.528,0.176
3,EXAMPLE_DATA_rfdiffusion_0033,rfdiffusion,3.8367,10.0,-36.0,0.600,0.000
4,EXAMPLE_DATA_rfdiffusion_0075,rfdiffusion,3.4900,6.0,-42.0,0.336,0.471
5,EXAMPLE_DATA_rfdiffusion_0198,rfdiffusion,3.2767,10.0,-33.0,0.640,0.000
6,EXAMPLE_DATA_rfdiffusion_0034,rfdiffusion,3.1033,8.0,-31.0,0.868,0.588
7,EXAMPLE_DATA_rfdiffusion_0173,rfdiffusion,2.9300,6.0,-34.0,0.356,0.059


## D3 (part 2) checklist
- [ ] Head-to-head: hit rate + interface-energy distribution per paradigm (figure `results/p10_headtohead.png`).
- [ ] **Occlusion** modeling of survivors (figure `results/p10_occlusion.png`); occluder count per paradigm.
- [ ] **Specificity vs human metalloenzymes** counter-test (min across the panel); selective count per paradigm.
- [ ] **Rim-vs-distal** ablation: distal binders bind but don't occlude — epitope choice drives inhibition.
- [ ] `results/top_candidates.csv`: top 10–20 each (occluding + selective), ready for the assay plan.
- [ ] Honest discussion: binding ≠ inhibition; the two paradigms' different failure modes (not just a winner).

**Next:** `05_validation_plan.ipynb` — the nitrocefin/carbapenem inhibition (IC50) plan.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation Plan — nitrocefin/carbapenem inhibition (IC50) + controls + β-lactam adjuvant

**Standard slot:** *validation plan.* **For Project 10 this means:** turn the top candidates into a
**costed, controlled wet-lab plan** for the assay that actually tests *inhibition* — an
**enzyme-kinetics INHIBITION assay** (**nitrocefin** chromogenic hydrolysis, or a **carbapenem-
hydrolysis** readout) measuring **IC50** — with the mandatory controls (**off-target
human-metalloenzyme** control, **scrambled-interface** negative, enzyme-only positive), an expression
strategy, and the **β-lactam-adjuvant** stretch (D4/D5).

A design that passes every filter and occludes the channel is a **hypothesis** — **binding ≠
inhibition**; the IC50 assay is what tests it. **No IC50 is generated in silico — that would be
fabricated.** Needs `results/top_candidates.csv` (notebook 04).

## Setup paths

In [32]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_10_ndm1_binder/notebooks


## 1 · Draft the inhibition-assay validation plan

Generate a plan card from the top candidates: the inhibition assay, controls, expression, timeline,
costed reagents. Fill the `<...>` from your own numbers; this is the deliverable other people will
actually read.

In [33]:
import pandas as pd, os

top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
n_top = len(top)
by_par = top.groupby("paradigm").size().to_dict() if n_top else {}

plan = f"""# NDM-1 Inhibitor (Binder) Validation Plan (Project 10 — by <your name>, <date>)

## Purpose (defensive anti-AMR)
INHIBIT NDM-1 (a carbapenem-hydrolyzing metallo-beta-lactamase) to RESTORE last-resort antibiotic
efficacy. Out of scope: enhancing resistance / pathogen fitness / stabilizing the enzyme.

## Candidates
Top {n_top} candidates carried forward ({by_par}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured. pae_interaction is confidence, occlusion is a
structural proxy -> BINDING != INHIBITION. There is NO in-silico IC50.

## Expression strategy
- Binders: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (40-80 aa) -> high yield expected.
- NDM-1: express the soluble construct; purify WITH Zn2+ in the buffer to keep the DI-ZINC site intact;
  confirm activity on nitrocefin BEFORE testing binders.

## Assays (go/no-go -> INHIBITION kinetics -> functional)
1. Go/no-go: express -> SDS-PAGE -> SEC (monodisperse binder?).
2. INHIBITION kinetics (the point): pre-incubate NDM-1 with a binder DILUTION SERIES, then add
   substrate and measure residual hydrolysis RATE:
     - nitrocefin (chromogenic cephalosporin; absorbance shift on hydrolysis), or
     - a carbapenem (imipenem/meropenem) hydrolysis readout (UV absorbance drop).
   Fit IC50 (and ideally K_i + mode: competitive / non-competitive / uncompetitive).
3. Stability: DSF (Tm). Deep (optional): co-crystal / cryo-EM of the binder-NDM-1 complex over the di-zinc site.

## Controls (MANDATORY)
- OFF-TARGET human-metalloenzyme control: run the SAME inhibition assay against a human Zn/metalloenzyme
  (e.g. carbonic anhydrase) -> the binder must NOT inhibit it (specificity / safety; mirrors nb-04 specificity).
- Negative (scrambled-interface): YOUR OWN top design with its interface residues scrambled/mutated
  -> must LOSE inhibition (cleanest specificity control).
- Enzyme-only / no-inhibitor positive: NDM-1 + substrate, no binder = 100% activity baseline; a known
  metallo-beta-lactamase chelator/inhibitor probe (e.g. EDTA / a captopril analogue) confirms the assay.

## (Stretch) beta-lactam ADJUVANT readout (the therapeutic point)
Checkerboard of binder x carbapenem (e.g. meropenem) in a resistant strain: does the binder RESTORE
the antibiotic's MIC (synergy / FIC index)? This is the defensive-anti-AMR proof-of-concept: the
binder makes a last-resort antibiotic work again.

## Realistic expectations
In-silico binder hit rates vary widely; the MAJORITY of in-silico hits fail experimentally, and
BINDING != INHIBITION. Expect to test many to find a few real inhibitors. Report the experimental hit
rate (and IC50s) honestly. Do NOT imply a working inhibitor or fabricate an IC50/K_i.

## Timeline + costed reagents (fill in)
- Gene synthesis ({n_top} binders + scrambled-interface negatives): $<...>, <...> weeks (IGSC-screened provider).
- NDM-1 + human off-target enzyme + nitrocefin/carbapenem substrate + plate reader time: $<...>.
- Personnel/instrument time: <...> weeks.

## Responsible research
Defensive anti-AMR: inhibit NDM-1 to restore carbapenem efficacy (in scope, MASTER_BLUEPRINT §7).
Out of scope: enhancing resistance / pathogen fitness. Gene synthesis via a biosecurity-screening
provider; wet lab under institutional biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> placeholders from your numbers.")
print(plan[:700], "...")

wrote results/validation_plan.md — fill the <...> placeholders from your numbers.
# NDM-1 Inhibitor (Binder) Validation Plan (Project 10 — by <your name>, <date>)

## Purpose (defensive anti-AMR)
INHIBIT NDM-1 (a carbapenem-hydrolyzing metallo-beta-lactamase) to RESTORE last-resort antibiotic
efficacy. Out of scope: enhancing resistance / pathogen fitness / stabilizing the enzyme.

## Candidates
Top 9 candidates carried forward ({'rfdiffusion': 9}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured. pae_interaction is confidence, occlusion is a
structural proxy -> BINDING != INHIBITION. There is NO in-silico IC50.

## Expression strategy
- Binders: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (40-80 aa) -> high yi ...


## 2 · Build the scrambled-interface negative controls

The single cleanest specificity control: take each top design and **scramble its interface residues**
(the positions contacting the NDM-1 rim) — it should **lose** inhibition. Generating these alongside
the real designs (same expression batch) makes the inhibition comparison airtight. Here we scaffold
the sequence-level scramble deterministically; on Colab, scramble the *interface* positions
specifically using the predicted contacts.

In [34]:
import random
import binder_tools as bt   # bt._hashints gives a DETERMINISTIC seed (Python's hash() is salted)

def scramble_interface(seq, frac=0.4, seed=0):
    """Deterministically shuffle a fraction of the sequence as a NEGATIVE-CONTROL stand-in.
    On Colab, scramble the predicted INTERFACE residues specifically (positions contacting the NDM-1 rim)."""
    rng = random.Random(seed)
    seq = list(seq)
    idx = list(range(len(seq)))
    rng.shuffle(idx)
    k = max(1, int(len(seq) * frac))
    chosen = idx[:k]
    vals = [seq[i] for i in chosen]
    rng.shuffle(vals)
    for i, v in zip(chosen, vals):
        seq[i] = v
    return "".join(seq)

import pandas as pd, os
top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
negs = []
if len(top) and "sequence" in top.columns:
    for _, r in top.iterrows():
        s = str(r.get("sequence", ""))
        if s:
            negs.append(dict(design_id=str(r["design_id"]) + "_SCRAM",
                             parent=r["design_id"], paradigm=r.get("paradigm"),
                             sequence=scramble_interface(s, seed=bt._hashints(r["design_id"]) % 10**6),
                             role="scrambled-interface negative control (must LOSE inhibition)"))
    pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
    print(f"wrote results/negative_controls.csv: {len(negs)} scrambled-interface negatives")
else:
    print("Run notebook 04 first to produce results/top_candidates.csv with sequences.")

wrote results/negative_controls.csv: 9 scrambled-interface negatives


## 3 · (Stretch) β-lactam adjuvant + affinity scaffold `[stretch]`

The therapeutic payoff is a **β-lactam adjuvant**: binder + carbapenem restoring the antibiotic's MIC
in a resistant strain (checkerboard / FIC index). And a predicted-affinity tool (Boltz-2) can
**prioritize** which top hits to test first. Use both for **relative ranking + caveats only** —
**never fabricate an IC50, K_i, K_D, or MIC**, and never present a prediction as a measurement.

In [35]:
# Scaffold ONLY. Do NOT invent IC50/K_i/K_D/MIC. On Colab:
#   - beta-lactam adjuvant: plan a binder x meropenem checkerboard in a blaNDM-1+ resistant strain;
#     report the FIC index / MIC shift from the WET-LAB experiment (not a model).
#   - affinity prioritization: pip install boltz; build the (binder, NDM-1) complex input; run boltz
#     predict; report the RELATIVE ranking of the top hits + heavy caveats (binding != inhibition).
# Pinned upstream (verify): https://github.com/jwohlwend/boltz
print("Stretch scaffold: beta-lactam adjuvant (restore antibiotic MIC) + affinity prioritization.")
print("Relative ranking + caveats only — NEVER a fabricated IC50/K_i/K_D/MIC. Binding != inhibition.")
print("Use it to PRIORITIZE which top hits to test first in the nitrocefin/carbapenem assay — not as evidence.")

Stretch scaffold: beta-lactam adjuvant (restore antibiotic MIC) + affinity prioritization.
Relative ranking + caveats only — NEVER a fabricated IC50/K_i/K_D/MIC. Binding != inhibition.
Use it to PRIORITIZE which top hits to test first in the nitrocefin/carbapenem assay — not as evidence.


## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: **nitrocefin/carbapenem INHIBITION (IC50)** assay, expression, timeline, costed reagents.
- [ ] Controls specified: **off-target human-metalloenzyme** control, **scrambled-interface** negative (`results/negative_controls.csv`), enzyme-only positive.
- [ ] (Stretch) β-lactam-adjuvant checkerboard (MIC restoration) + Boltz-2 affinity for relative ranking — no fabricated IC50/MIC.
- [ ] Honest framing: every design is a hypothesis; **binding ≠ inhibition** until the IC50 assay; report the experimental hit rate.
- [ ] Defensive-anti-AMR framing throughout (inhibit NDM-1, restore carbapenems; never enhance resistance/fitness).
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a defensive anti-AMR binder/inhibitor campaign, honestly reported.